**`harmonize_buildings`**

Create a spine for footprints (identifiers for all observed footprints)

The current version is specific to the U.S. context (not generalized)

# Configure

In [ ]:
import argparse
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from openplaces.api import get_admin, read_entities
from openplaces.geo.ids import (
    add_openlocationcode_index,
)
from openplaces.geo.polygon import (
    get_areas,
    overlay_polygons,
    resolve_overlapping_polygons,
)
from openplaces.io import save_parquet, share
from openplaces.io.transform import make_index_unique, remap
from openplaces.path import external_path, path, share_path
from openplaces.recipe import get_output_path
from openplaces.viz import show_building
from openplaces.viz.tabulation import plot_tabulation

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(
    description='Harmonize building data from multiple recipes'
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-RI")',
    nargs='*',
)
parser.add_argument(
    '--show_examples',
    help='Make maps of examples (using matplotlib)',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Example 1: Brunswick, NC (hurricane risk case)
    '--admin_ids US-NC-BS '  # Brunswick
    # '--admin_ids US-NC-NE '  # New Hanover (Wilmington)
    # '--admin_ids US-NC-PD '  # Pender
    # '--admin_ids US-NC-WK '  # Wake
    # '--admin_ids US-NC-BL '
    # '--admin_ids US-NC-CU '  # Duplicate building ID
    # Example 2: Buncombe, NC (fluvial flood risk case)
    # '--admin_ids US-NC-BO'
    # Example 3: Jefferson and Harris, NC (hurricane risk cases)
    # '--admin_ids US-TX-JE US-TX-RR'
    # '--show_examples '
    '--verbose'
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

# Prepare data

In [ ]:
# Future start of loop
for admin_id in args.admin_ids:
    break

print(admin_id)

# For visualization purposes
admin3 = get_admin(admin_id, level=3, geom=True)
admin3

## Prepare footprints

In [ ]:
footprints_obm = read_entities('building-obm-2025', admin_id, geom=True)
footprints_microsoft = read_entities('US_building-microsoft-v2', admin_id, geom=True)
footprints_fema = read_entities('US_building-fema-2023', admin_id, geom=True)
nsi = read_entities('US_building-nsi-2022', admin_id, geom=True)
if admin_id.startswith('US-NC'):
    footprints_nc = read_entities('US-NC_building-ncdps-2023', admin_id, geom=True)

In [ ]:
# Hesam Soleimani's first inventory
BUILDINGS_CHEER_PATH = external_path(
    'US-NC', 'building-cheer-v0', filename='Inventory_v0_NC.parquet'
)
# Get county FIPS code
county_fips = get_admin(admin_id)['admin3_id_admin1'].values[0]
footprints_cheer = gpd.read_parquet(
    BUILDINGS_CHEER_PATH, filters=[('county', '==', county_fips)]
).set_index('bid')

In [ ]:
# Remap building groups
nsi = remap(nsi, 'US_building-nsi-2022_purpose-subgroup-remap')
footprints_fema = remap(footprints_fema, 'US_building-fema-2023_purpose-subgroup-remap')

In [ ]:
# Calculate polygon areas
footprints_microsoft['m2'] = get_areas(footprints_microsoft, 'm2')
footprints_fema['m2'] = get_areas(footprints_fema, 'm2')

## Prepare parcels

In [ ]:
parcels = read_entities('US-NC_parcel-nconemap-2025', admin_id, geom=True)

# US-NC: remove parcels that are identical in everything but the
# `source_deed` these seem to be duplicates from records going over
# multiple pages
mask_duplicates = parcels.drop(columns=['source_deed']).duplicated()
parcels = parcels[~mask_duplicates].copy()

parcels['purpose_groups'] = pd.Categorical(
    parcels['purpose_group'].astype(str).fillna('n/a')
    + ' | '
    + parcels['purpose_subgroup'].astype(str).fillna('n/a')
)

# Identify identical polygons
parcels['has_duplicate_geometry'] = parcels['geo_id'].duplicated(keep=False)
if parcels['has_duplicate_geometry'].any():
    warnings.warn('Duplicate `geo_id` values found after dropping duplicate parcels.')
parcels['ha'] = get_areas(parcels, 'ha')

# Select unique parcels
parcel_polygons = parcels[~parcels['geo_id'].duplicated()][
    ['geometry', 'geo_id', 'ha', 'has_duplicate_geometry']
]
# Use 'geo_id' as index
parcel_polygons.index = parcel_polygons['geo_id'].rename('parcel_id')

# Join key indicators
parcels_aggregated = parcels.groupby('geo_id').agg(
    {
        'purpose_group': ' + '.join,
        'purpose_subgroup': ' + '.join,
        'purpose_groups': ' + '.join,
        'improvement_value': 'sum',
        'year_built': 'mean',
    }
)
parcels_aggregated['n_parcels'] = parcels.groupby('geo_id').size()
parcel_polygons = parcel_polygons.join(parcels_aggregated)

parcel_polygons['improvement_value_per_ha'] = (
    parcel_polygons['improvement_value'] / parcel_polygons['ha']
)

In [ ]:
if args.verbose:
    print(f'{len(footprints_obm):,d} footprints by OpenBuildingMap')
    print(f'{len(footprints_microsoft):,d} footprints by Microsoft')
    print(f'{len(footprints_fema):,d} footprints by FEMA')
    if admin_id.startswith('US-NC'):
        print(f'{len(footprints_nc):,d} dwelling footprints by NC-DPS')
    print(f'{len(nsi):,d} dwellings by NSI (points)')
    print(f'{len(footprints_cheer):,d} footprints in V0 inventory (CHEER)')
    print(f'{len(parcels):,d} parcels')

# Create building spine
1. Keep OpenBuildingMap footprints
2. Add Microsoft footprints that don't overlap
3. Add FEMA footprints that don't overlap
4. Add NSI points located outside of footprints
5. Add parcel-level information

In [ ]:
FOOTPRINT_OVERLAP_IOU_MAX = 0.02

## Keep OpenBuildingMap footprints

In [ ]:
footprints_obm['source'] = 'obm'
footprints = footprints_obm[['geometry', 'source']]

## Add Microsoft footprints that don't overlap
Allow only minor overlaps (geolocation errors)

In [ ]:
footprints_obm_microsoft = overlay_polygons(
    get_output_path('building-obm-2025', admin_id),
    get_output_path('US_building-microsoft-v2', admin_id),
    suffixes=('_obm', '_microsoft'),
    iou=True,
).sort_values('iou', ascending=False)
# footprints_obm_microsoft

In [ ]:
# Accepted size of overlap understood as identifying separate footprints
footprint_ids_microsoft = footprints_obm_microsoft[
    footprints_obm_microsoft['iou'].gt(FOOTPRINT_OVERLAP_IOU_MAX)
].index.get_level_values('footprint_id_microsoft')

footprints_microsoft_non_overlapping = footprints_microsoft[
    ~footprints_microsoft.index.isin(footprint_ids_microsoft)
]
footprints_microsoft_non_overlapping['source'] = 'microsoft'

footprints = pd.concat(
    [footprints, footprints_microsoft_non_overlapping[['geometry', 'source']]]
).sort_index()

## Add FEMA footprints that don't overlap

In [ ]:
# Using get_intersection_over_union so we don't have to save to disc
footprints_hybrid_fema = overlay_polygons(
    footprints, footprints_fema, suffixes=('_hybrid', '_fema'), iou=True
).sort_values('iou', ascending=False)
footprints_hybrid_fema

In [ ]:
# Accepted size of overlap understood as identifying separate footprints
footprint_ids_fema = footprints_hybrid_fema[
    footprints_hybrid_fema['iou'].gt(FOOTPRINT_OVERLAP_IOU_MAX)
].index.get_level_values('footprint_id_fema')

footprints_fema_non_overlapping = footprints_fema[
    ~footprints_fema.index.isin(footprint_ids_fema)
]
footprints_fema_non_overlapping['source'] = 'fema'

footprints = pd.concat(
    [footprints, footprints_fema_non_overlapping[['geometry', 'source']]]
).sort_index()

footprints['source'] = pd.Categorical(
    footprints['source'], categories=['obm', 'microsoft', 'fema'], ordered=True
)

## Add footprints from parcels
### Identify parcels with footprints

In [ ]:
# Columns to keep in harmonized dataset
FOOTPRINT_PARCEL_COLS = [
    'footprint_id',
    'parcel_id',
    'area_intersection_m2',
    'iou',
    'area_intersection_m2_inner',
    'fraction_of_largest',
]

FOOTPRINT_PARCEL_AREA_INTERSECTION_M2_MIN = 10

FOOTPRINT_PARCEL_MIN_FRACTION_OF_LARGEST = 1 / 6

In [ ]:
footprints_on_parcels = overlay_polygons(
    footprints,
    parcel_polygons,
    suffixes=('_building', '_parcel'),
    how='identity',
    iou=True,
    geom=True,
)
print(len(footprints_on_parcels))

### Keep unique parcels per footprint

In [ ]:
mask_footprints_multiparcel = footprints_on_parcels.index.get_level_values(
    'footprint_id'
).duplicated(keep=False)

cols = [v for v in FOOTPRINT_PARCEL_COLS if v in footprints_on_parcels]

# Initiate crosswalk (footprint_id > parcel_id)
footprints_single_parcel = (
    footprints_on_parcels[~mask_footprints_multiparcel]
    .reset_index()
    .set_index('footprint_id')[['parcel_id'] + cols]
)
link = np.where(
    footprints_single_parcel['parcel_id'].notnull(), 'unique parcel', 'no parcel'
)
footprints_single_parcel.insert(1, 'link', link)
# footprints_single_parcel

### Add unique parcels per footprint after removing small neighbors

In [ ]:
# Compute inner buffer (to identify slivers)
footprints_multiparcel = footprints_on_parcels[mask_footprints_multiparcel].copy()
footprints_multiparcel['fraction_of_largest'] = (
    footprints_multiparcel['area_intersection_m2']
    / footprints_multiparcel.groupby('footprint_id')['area_intersection_m2'].transform(
        'max'
    )
).round(3)
footprints_multiparcel_trimmed = footprints_multiparcel.query(
    f'fraction_of_largest >= {FOOTPRINT_PARCEL_MIN_FRACTION_OF_LARGEST} '
    f'and area_intersection_m2 >= {FOOTPRINT_PARCEL_AREA_INTERSECTION_M2_MIN}'
)
# footprints_multiparcel[['iou', 'area_intersection_m2', 'fraction_of_largest']]

In [ ]:
mask_footprints_multiparcel_to_split = (
    footprints_multiparcel_trimmed.index.get_level_values('footprint_id').duplicated(
        keep=False
    )
)

cols = [v for v in FOOTPRINT_PARCEL_COLS if v in footprints_multiparcel]

# Add newly resolved unique parcels to crosswalk
footprints_single_parcel_trimmed = (
    footprints_multiparcel_trimmed[~mask_footprints_multiparcel_to_split]
    .reset_index()
    .set_index('footprint_id')[['parcel_id'] + cols]
)
link = np.where(
    footprints_single_parcel_trimmed['parcel_id'].notnull(),
    'unique parcel',
    'no parcel',
)
footprints_single_parcel_trimmed.insert(1, 'link', link + ' (dropping small neighbor)')
footprints_single_parcel = pd.concat(
    [
        footprints_single_parcel.drop(
            set(footprints_single_parcel.index)
            & set(footprints_single_parcel_trimmed.index)
        ),
        footprints_single_parcel_trimmed,
    ]
).sort_index()

# footprints_single_parcel

### Assign remaining footprints to multiple parcels

In [ ]:
cols = [v for v in FOOTPRINT_PARCEL_COLS if v in footprints_multiparcel_trimmed]

footprints_multiparcel_to_split = footprints_multiparcel_trimmed[
    mask_footprints_multiparcel_to_split
][cols + ['geometry']]
footprints_multiparcel_to_split.insert(0, 'link', 'multi-parcel footprint')

In [ ]:
footprint_parcels = pd.concat(
    [
        footprints_single_parcel.reset_index().set_index(['footprint_id', 'parcel_id']),
        footprints_multiparcel_to_split.drop(columns='geometry'),
    ]
).sort_index()

In [ ]:
if args.show_examples:
    # FOOTPRINT_ID = '8753WXJF+4Q4'  # manufactured home that has been moved
    # footprint_id = FOOTPRINT_ID

    # Display multi-parcel footprint
    footprint_id = footprints_multiparcel_to_split.sample().index[0][0]
    print('footprint_id:', footprint_id)

    fig, ax = show_building(
        location=footprints.loc[[footprint_id]],
        geodatasets={
            'parcels': parcels,
            'footprints_fema': footprints_fema,
            'nsi': nsi,
            'footprints_microsoft': footprints_microsoft,
        },
        radius=50,
        return_fig_ax=True,
    )
    # footprints_multiparcel_to_split.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(
    # ax=ax, color='red'
    # )
    ax.set_title('Example: multi-parcel footprint')

### Infer buildings from parcels
Currently only including footprints - no NSI points added
#### Compute parcel statistics for known footprints

In [ ]:
BUILDING_FROM_PARCELS_COLS = [
    'purpose_group',
    'improvement_value_per_ha',
    'has_duplicate_geometry',
]

N_FOOTPRINTS_MEAN_PER_GROUP_MIN = 0.2

IMPROVEMENT_VALUE_PER_HA_QUANTILE = 0.05

In [ ]:
# Join parcel data to building-parcel links to compute purpose-group
# specific statistics
footprint_parcel_data = footprint_parcels[['link']].join(
    parcel_polygons[~parcel_polygons['has_duplicate_geometry']][
        BUILDING_FROM_PARCELS_COLS
    ],
    on='parcel_id',
    how='inner',
)

In [ ]:
n_footprints_per_group = (
    (
        footprint_parcel_data['purpose_group'].value_counts()
        / parcel_polygons['purpose_group'].value_counts()
    )
    .fillna(0)
    .rename('n_footprints_mean')
)
n_footprints_per_group.sort_values()

In [ ]:
imp_val_quant_column = f'improvement_value_per_ha_q{IMPROVEMENT_VALUE_PER_HA_QUANTILE}'

# Get improvement value for 1-1 matching footprints.
improvement_value_per_ha_q_by_parcel_group = (
    (
        footprint_parcel_data.sample(frac=1)
        .reset_index()
        .drop_duplicates('parcel_id', keep=False)
        .groupby('purpose_group')['improvement_value_per_ha']
        .quantile((IMPROVEMENT_VALUE_PER_HA_QUANTILE, 0.5))
    )
    .unstack()
    .rename(
        columns={
            IMPROVEMENT_VALUE_PER_HA_QUANTILE: imp_val_quant_column,
            0.5: 'improvement_value_per_ha_median',
        }
    )
)

improvement_value_per_ha_q_by_parcel_group.sort_values(
    'improvement_value_per_ha_median'
).map('$ {0:,.0f}'.format)

#### Create buildings from parcels

In [ ]:
mask_parcels_without_footprints = ~parcel_polygons.index.isin(
    footprint_parcels.index.get_level_values('parcel_id').unique()
)

parcel_candidates = (
    parcel_polygons[mask_parcels_without_footprints][
        [
            'purpose_group',
            'improvement_value',
            'improvement_value_per_ha',
            'has_duplicate_geometry',
            'geometry',
        ]
    ]
    .join(improvement_value_per_ha_q_by_parcel_group, on='purpose_group')
    .join(n_footprints_per_group, on='purpose_group')
)

# Lowest improvement value to mark new footprints: low quantile of
# per-area (hectare) improvement value among all parcels with non-zero
# improvement value (thresholds will vary by county).
imp_value_per_ha_min = (
    parcel_polygons[
        parcel_polygons.index.isin(
            footprint_parcels.index.get_level_values('parcel_id').unique()
        )
    ]
    .query('improvement_value_per_ha > 0')['improvement_value_per_ha']
    .quantile(IMPROVEMENT_VALUE_PER_HA_QUANTILE)
)
print(f'Buildings inferred if improvement value > $ {imp_value_per_ha_min:,.0f}.')

mask_parcels_with_inferred_footprints = parcel_candidates['n_footprints_mean'].gt(
    N_FOOTPRINTS_MEAN_PER_GROUP_MIN
) & parcel_candidates['improvement_value_per_ha'].gt(
    parcel_candidates[imp_val_quant_column].div(2).clip(lower=imp_value_per_ha_min)
)

In [ ]:
parcels_with_inferred_footprints = parcel_candidates[
    mask_parcels_with_inferred_footprints
]

footprints_from_parcels = add_openlocationcode_index(
    parcels_with_inferred_footprints[['geometry']].reset_index(), name='footprint_id'
)

# Remove duplicates (NSI points whose OLC didn't link to parcel)
footprints_from_parcels = footprints_from_parcels[
    ~footprints_from_parcels.index.isin(footprints.index)
]

footprints_from_parcels['source'] = 'parcel'
footprints = pd.concat(
    [footprints, footprints_from_parcels[['geometry', 'source']]]
).sort_index()

mask_duplicate_index_values = footprints.index.duplicated(keep=False)
if mask_duplicate_index_values.any():
    warnings.warn(
        f'Created {mask_duplicate_index_values.sum()} duplicate `building_ids`.'
    )
    footprints = make_index_unique(footprints, sort_duplicates_by_area=True)

### Remove overlaps

In [ ]:
footprints = resolve_overlapping_polygons(footprints, keep=False)

# Infer attributes from NSI

## Join NSI to footprints on parcels
Cleanest - NSI used parcel data, so this ensures parcel data is used for allocating points to parcels

In [ ]:
NSI_FILTER_COLS = [
    'purpose_subgroup',
    'openplaces_group',
    'structure_value',
    'year_built_block_median',
    'source',
    'area_sqft',
    'n_stories',
]
FOOTPRINT_PARCEL_FILTER_COLS = []
nsi_on_footprints_on_parcels = (
    gpd.sjoin(
        nsi[NSI_FILTER_COLS + ['geometry']].rename(
            columns={k: f'{k}_nsi' for k in NSI_FILTER_COLS}
        ),
        resolve_overlapping_polygons(footprints_on_parcels, keep=False)[
            FOOTPRINT_PARCEL_FILTER_COLS + ['geometry']
        ],
    )
    .drop(columns='geometry')
    .sort_values(['source_nsi', 'structure_value_nsi'], ascending=[True, False])
)

mask_footprint_parcels_overlap = nsi_on_footprints_on_parcels.index.duplicated(
    keep=False
)
if mask_footprint_parcels_overlap.any():
    print(
        f'{mask_footprint_parcels_overlap.sum()} NSI points '
        f'({mask_footprint_parcels_overlap.mean():.3%} of linked) '
        f'are linked to more than one footprint-parcel overlap.\n'
        '-> Kept first occurrence and removed duplicates.\n'
    )
    nsi_on_footprints_on_parcels = nsi_on_footprints_on_parcels[
        ~nsi_on_footprints_on_parcels.index.duplicated()
    ].copy()

mask_footprint_parcel_duplicated = nsi_on_footprints_on_parcels[
    ['footprint_id', 'parcel_id']
].duplicated(keep=False)
if mask_footprint_parcel_duplicated.any():
    print(
        f'{mask_footprint_parcel_duplicated.sum():,d} NSI points '
        f'({mask_footprint_parcel_duplicated.mean():.1%} of linked) '
        'share same footprint-parcel overlap.'
    )

In [ ]:
# Demonstrate how ESRI NSI entries are often wrong
# if mask_footprint_parcel_duplicated.any() and args.show_examples:
#     for i in range(3):
#         sample = nsi_on_footprints_on_parcels[mask_footprint_parcel_duplicated].sample()
#         building_id_nsi = sample.index[0]
#         footprint_id, parcel_id = sample[['footprint_id', 'parcel_id']].loc[building_id_nsi]

#         FOOTPRINT_PARCEL_ID = [footprint_id, parcel_id]
#         print(', '.join(FOOTPRINT_PARCEL_ID))

#         fig, ax = show_building(
#             nsi.loc[[building_id_nsi]],
#             geodatasets={
#                 'parcels': parcels,
#                 'nsi': nsi,
#                 'footprints_local': footprints_obm,
#             },
#             # radius=200,
#             return_fig_ax=True,
#         )
#         footprints.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(ax=ax, color='red')
#         ax.set_title(f'Multiple NSI points share footprint-parcel link {building_id_nsi}')
#         plt.show()

In [ ]:
mask_duplicate_footprints_to_drop = (
    mask_footprint_parcel_duplicated
    & nsi_on_footprints_on_parcels['source_nsi'].isin(['ESRI', 'HAZUS/NSI-2015'])
    & (
        nsi_on_footprints_on_parcels.groupby(['footprint_id', 'parcel_id'], sort=False)[
            'source_nsi'
        ]
        .transform('first')
        .eq('Parcel')
    )
)
print(
    f'{mask_duplicate_footprints_to_drop.sum():,d} '
    f'({mask_duplicate_footprints_to_drop.mean():.1%})'
    ' are ESRI/HAZUS vs. Parcel duplicates and will be dropped.'
)

### Cases to drop

In [ ]:
# if mask_duplicate_footprints_to_drop.any() and args.show_examples:
#     for i in range(3):
#         sample = nsi_on_footprints_on_parcels[mask_duplicate_footprints_to_drop].iloc[[i]]
#         building_id_nsi = sample.index[0]
#         footprint_id, parcel_id = sample[['footprint_id', 'parcel_id']].loc[building_id_nsi]

#         FOOTPRINT_PARCEL_ID = [footprint_id, parcel_id]
#         print(', '.join(FOOTPRINT_PARCEL_ID))

#         fig, ax = show_building(
#             nsi.loc[[building_id_nsi]],
#             geodatasets={
#                 'parcels': parcels,
#                 'nsi': nsi,
#                 'footprints_fema': footprints_fema,
#                 'footprints_local': footprints_obm,
#             },
#             # radius=200,
#             return_fig_ax=True,
#         )
#         footprints.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(ax=ax, color='red')
#         ax.set_title(f'Dropping ESRI: NSI duplicate @ {building_id_nsi}')
#         plt.show()

### Cases to explore
Most seem industrial - is this relevant for SFH?

In [ ]:
mask_unresolved_footprint_parcel_duplicates = (
    mask_footprint_parcel_duplicated & ~mask_duplicate_footprints_to_drop
)

In [ ]:
# if mask_unresolved_footprint_parcel_duplicates.any() and args.show_examples:
#     for i in range(3):
#         sample = nsi_on_footprints_on_parcels[mask_unresolved_footprint_parcel_duplicates].iloc[[i]]
#         building_id_nsi = sample.index[0]
#         footprint_id, parcel_id = sample[['footprint_id', 'parcel_id']].loc[building_id_nsi]

#         FOOTPRINT_PARCEL_ID = [footprint_id, parcel_id]
#         print(', '.join(FOOTPRINT_PARCEL_ID))

#         fig, ax = show_building(
#             nsi.loc[[building_id_nsi]],
#             geodatasets={
#                 'parcels': parcels,
#                 'nsi': nsi,
#                 'footprints_fema': footprints_fema,
#                 'footprints_local': footprints_obm,
#             },
#             # radius=200,
#             return_fig_ax=True,
#         )
#         footprints.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(ax=ax, color='red')
#         ax.set_title(f'Multiple NSI points share footprint-parcel link {building_id_nsi}')
#         plt.show()

## Join unlinked NSI to parcels

In [ ]:
mask_unlinked_nsi = ~nsi.index.isin(nsi_on_footprints_on_parcels.index)
print(
    f'{mask_unlinked_nsi.sum():,d} NSI points ({mask_unlinked_nsi.mean():.1%}) '
    'not linked to footprint-parcels.'
)

nsi_on_parcels = gpd.sjoin(
    nsi[mask_unlinked_nsi][NSI_FILTER_COLS + ['geometry']].rename(
        columns={k: f'{k}_nsi' for k in NSI_FILTER_COLS}
    ),
    parcel_polygons[['geometry']],
    how='left',
).drop(columns='geometry')

mask_parcels_overlap = nsi_on_parcels.index.duplicated(keep=False)
if mask_parcels_overlap.any():
    print(
        f'{mask_parcels_overlap.sum()} NSI points '
        f' are linked to more than one parcel '
        f'({mask_parcels_overlap.mean():.3%} of linked). '
        'Kept first, removed duplicates.'
    )
    nsi_on_parcels = nsi_on_parcels[~nsi_on_parcels.index.duplicated()].copy()

mask_footprint_parcel_duplicated = nsi_on_parcels['parcel_id'].duplicated(keep=False)
if mask_footprint_parcel_duplicated.any():
    print(
        f'{mask_footprint_parcel_duplicated.sum():,d} NSI points '
        f'({mask_footprint_parcel_duplicated.mean():.1%} of linked) '
        'share the same parcel.'
    )

In [ ]:
nsi_linked = pd.concat(
    [nsi_on_footprints_on_parcels[~mask_duplicate_footprints_to_drop], nsi_on_parcels]
)

## Infer building types

In [ ]:
PARCEL_COLS = [
    'purpose_group',
    'purpose_subgroup',
    'purpose_groups',
    'improvement_value',
    'year_built',
]

dwellings_nsi = nsi_linked.sort_index().join(
    parcels[PARCEL_COLS].rename(columns={k: k + '_parcel' for k in PARCEL_COLS}),
    on='parcel_id',
    lsuffix='_point',
)
# Combine 'purpose_group_parcel' and 'purpose_subgroup_parcel'
# for more precise inference of NSI categories

In [ ]:
if args.show_examples:
    plot_tabulation(
        dwellings_nsi,
        x_cat='openplaces_group_nsi',
        x_max_n=10,
        y_cat='purpose_groups_parcel',
        y_max_n=15,
    )

In [ ]:
PURPOSE_GROUP_COLUMN = 'purpose_groups_parcel'
PURPOSE_GROUP_NEW_COLUMN = 'openplaces_group_nsi'
# PURPOSE_GROUP_NEW_COLUMN = 'purpose_subgroup_nsi'

counts = dwellings_nsi[[PURPOSE_GROUP_COLUMN, PURPOSE_GROUP_NEW_COLUMN]].value_counts()
fractions = (
    counts.div(counts.groupby(PURPOSE_GROUP_COLUMN).sum()).rename('fraction').round(3)
)
inferred_groups = (
    pd.concat([counts, fractions], axis=1)
    .reset_index()
    .sort_values([PURPOSE_GROUP_COLUMN, 'count'], ascending=[True, False])
    .drop_duplicates(PURPOSE_GROUP_COLUMN)
    .set_index(PURPOSE_GROUP_COLUMN)  # ['openplaces_group_nsi']
).sort_values('count', ascending=False)
inferred_groups.head(20)

## Attribute NSI building types & counts.


In [ ]:
footprints['n_nsi'] = nsi_linked.groupby('footprint_id').size()
footprints['n_nsi'] = footprints['n_nsi'].fillna(0)

In [ ]:
footprint_purpose_group_nsi = (
    nsi_linked.groupby(['footprint_id', 'purpose_subgroup_nsi'])['structure_value_nsi']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

footprints['purpose_subgroup_nsi'] = footprint_purpose_group_nsi.drop_duplicates(
    'footprint_id'
).set_index('footprint_id')['purpose_subgroup_nsi']

footprints['purpose_groups_all_nsi'] = footprint_purpose_group_nsi.groupby(
    'footprint_id'
)['purpose_subgroup_nsi'].apply(' + '.join)

In [ ]:
footprints = footprints.join(
    nsi_linked.groupby('footprint_id').agg(
        {'structure_value_nsi': 'sum', 'year_built_block_median_nsi': 'mean'}
    )
)

In [ ]:
footprint_purpose_group_nsi = (
    nsi_linked.groupby(['footprint_id', 'openplaces_group_nsi'])['structure_value_nsi']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

footprints['openplaces_group_nsi'] = footprint_purpose_group_nsi.drop_duplicates(
    'footprint_id'
).set_index('footprint_id')['openplaces_group_nsi']

footprints['openplaces_group_all_nsi'] = footprint_purpose_group_nsi.groupby(
    'footprint_id'
)['openplaces_group_nsi'].apply(' + '.join)

## Attribute parcel-derived building types & counts

In [ ]:
# Compute fraction of parcel-footprint overlaps by parcel
footprints_on_parcels['area_fraction'] = footprints_on_parcels[
    'area_intersection_m2'
] / footprints_on_parcels.groupby('parcel_id')['area_intersection_m2'].transform('sum')

In [ ]:
# Attribute data from footprints on parcels
mask_footprints_with_parcel = footprints_on_parcels.index.get_level_values(
    'parcel_id'
).notnull()
footprints_with_parcel_data = footprints_on_parcels[mask_footprints_with_parcel][
    ['area_intersection_m2', 'area_fraction']
].join(parcels[PARCEL_COLS])

# Identify purpose group of largest footprint-parcel overlap by footprint
footprints['n_footprint_parcel'] = footprints_with_parcel_data.groupby(
    'footprint_id'
).size()
footprints['n_footprint_parcel'] = footprints['n_footprint_parcel'].fillna(0)
footprint_purpose_group_areas = (
    footprints_with_parcel_data.groupby(['footprint_id', 'purpose_groups'])[
        'area_intersection_m2'
    ]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
footprints['purpose_groups_parcel'] = footprint_purpose_group_areas.drop_duplicates(
    'footprint_id'
).set_index('footprint_id')['purpose_groups']
footprints['purpose_groups_all_parcel'] = footprint_purpose_group_areas.groupby(
    'footprint_id'
)['purpose_groups'].apply(' + '.join)

In [ ]:
PARCEL_COLS_TO_SPLIT = ['improvement_value']
for col in PARCEL_COLS_TO_SPLIT:
    footprints_with_parcel_data[col] = (
        footprints_with_parcel_data[col]
        .mul(footprints_with_parcel_data['area_fraction'])
        .round(2)
    )

In [ ]:
footprints = footprints.join(
    footprints_with_parcel_data.groupby('footprint_id')
    .agg({'improvement_value': 'sum', 'year_built': 'mean'})
    .rename(
        columns={
            'improvement_value': 'improvement_value_parcel',
            'year_built': 'year_built_parcel',
        }
    )
)

In [ ]:
# Attribute data from parcels without footprints
mask_parcel_footprints = footprints['source'].eq('parcel')
_footprints_from_parcels = (
    footprints_from_parcels[['parcel_id']]
    .join(parcel_polygons[PARCEL_COLS], on='parcel_id')
    .rename(
        columns={
            'purpose_groups': 'purpose_groups_parcel',
            'improvement_value': 'improvement_value_parcel',
            'year_built': 'year_built_parcel',
        }
    )
)
_footprints_from_parcels['n_footprint_parcel'] = 1
cols = [v for v in footprints if v in _footprints_from_parcels]
footprints.loc[mask_parcel_footprints, cols] = _footprints_from_parcels[cols]
del _footprints_from_parcels

In [ ]:
footprints = footprints.join(
    inferred_groups['openplaces_group_nsi'].rename('openplaces_group_parcel'),
    on='purpose_groups_parcel',
)

# Show examples

In [ ]:
N_SHOW = 2

if args.show_examples:
    if admin_id == 'US-NC-BS':
        PURPOSE_GROUPS = 'RES RU ACR | '
        PURPOSE_GROUPS = 'MH SUBDIV | SINGLE WIDE AS REAL PROPERTY'
        PURPOSE_GROUPS = 'RES RU ACR | MODULAR'
        PURPOSE_GROUPS = 'TOWNHOME | TOWNHOME'
    else:
        raise ValueError('No purpose group defined.')

    dwellings_nsi_subgroup = dwellings_nsi.query(
        f"purpose_groups_parcel == '{PURPOSE_GROUPS}'"
    )
    for i in range(N_SHOW):
        sample = dwellings_nsi_subgroup.sample()
        building_id_nsi = sample.index[0]
        footprint_id, parcel_id = sample[['footprint_id', 'parcel_id']].loc[
            building_id_nsi
        ]

        print(f'footprint: {footprint_id}, parcel: {parcel_id}')

        fig, ax = show_building(
            nsi.loc[[building_id_nsi]],
            geodatasets={
                'parcels': parcels,
                'nsi': nsi,
                # 'footprints_fema': footprints_fema,
                'footprints_local': footprints_obm,
            },
            # radius=200,
            return_fig_ax=True,
        )
        if pd.notnull(footprint_id):
            footprints.loc[[footprint_id]].to_crs('epsg:3857').boundary.plot(
                ax=ax, color='red'
            )
        ax.set_title(f'Parcel purpose groups: {PURPOSE_GROUPS}')
        plt.show()

# Add columns for mapping

In [ ]:
import warnings

with warnings.catch_warnings():
    footprints['m2'] = get_areas(footprints, unit='m2')
    footprints['improvement_value_per_area_parcel'] = (
        footprints['improvement_value_parcel'] / footprints['m2']
    )
    footprints['structure_value_per_area_nsi'] = (
        footprints['structure_value_nsi'] / footprints['m2']
    )

In [ ]:
a = footprints['openplaces_group_parcel']
b = footprints['openplaces_group_nsi']

both = a.notna() & b.notna()
same = both & (a == b)

footprints['openplaces_group_parcel_nsi'] = np.select(
    [same, both, a.notna(), b.notna()],
    [a, 'nsi: ' + a + ' | parcel: ' + b, a, b],
    default='',
)

# Save footprints

In [ ]:
save_parquet(footprints, path(admin_id, 'building-openplaces-2026'))

In [ ]:
share(
    footprints,
    share_path(admin_id, 'building-openplaces-2026'),
    'share/2026/cheer',
    delete_original=False,
)

In [ ]:
assert False, 'This marks the end of a flattened `for` loop.'

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/.../'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Loop script

In [ ]:
CHEER_ADMIN3_IDS = 'US-NC-BA US-NC-BT US-NC-BL US-NC-BS US-NC-CD US-NC-CE US-NC-CW US-NC-CM US-NC-CN US-NC-CU US-NC-CI US-NC-DE US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL US-NC-HT US-NC-HD US-NC-HO US-NC-HE US-NC-JH US-NC-JN US-NC-LN US-NC-MR US-NC-NA US-NC-NE US-NC-NO US-NC-ON US-NC-PM US-NC-PU US-NC-PD US-NC-PQ US-NC-PI US-NC-RB US-NC-SP US-NC-SC US-NC-TY US-NC-WK US-NC-WR US-NC-WI US-NC-WY US-NC-WO'.split()

In [ ]:
for admin3_id in CHEER_ADMIN3_IDS:
    print(admin3_id)

    args_list_cheer = ['--admin_ids'] + [admin3_id] + ['--verbose']

    print(' '.join(args_list_cheer))
    test_script(*args_list_cheer)